In [63]:
from sklearn import datasets
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import normalized_mutual_info_score
import numpy as np

#Loading in the iris dataset
iris = datasets.load_iris()
X = iris.data[:, :4]  # Exclude the last column (class label)

#Scale each attribute so that it is within range 0 to 1
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

#Find_Attractor function of the DENCLUE algorithm
def find_attractor(x, X, h, epsilon):
    delta = float('inf')
    while delta > epsilon:
        distances = np.sqrt(((X - x) ** 2).sum(axis=1))
        weights = (1 / (h * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((distances / h)) ** 2)
        next_x = (weights[:, np.newaxis] * X).sum(axis=0) / weights.sum()
        delta = np.sqrt(((next_x - x) ** 2).sum())
        x = next_x
    return x

#DENCLUE algorithm
def denclue(X, h, epsilon, theta):
    attractors = []
    for x in X:
        attractor = find_attractor(x, X, h, epsilon)
        is_merged = False
        for a in attractors:
            if np.linalg.norm(attractor - a) < theta:
                is_merged = True
                break
        if not is_merged:
            attractors.append(attractor)
    
    #Assigns points to clusters depending on the closest attractor
    point_labels = np.zeros(X.shape[0], dtype=int)
    for i, x in enumerate(X):
        distances = [np.linalg.norm(x - attractor) for attractor in attractors]
        point_labels[i] = np.argmin(distances)
    
    #Finds cluster sizes
    cluster_sizes = [np.sum(point_labels == i) for i in range(len(attractors))]
    return point_labels, cluster_sizes, attractors


In [64]:
##Initializing values for h, epsilon, theta, and the true_labels of the iris set
h = 0.1
epsilon = 0.001
theta_values = np.linspace(0.01, 1, 50)  #50 thresholds within the range 0.01 and 1
true_labels = iris.target

#Prints information on all cluster sets and calls functions
best_nmi = -1
cluster_size_3_info = []
print("Information all on cluster sets:")
for theta in theta_values:
    labels, cluster_sizes, attractors = denclue(X_scaled, h, epsilon, theta)
    #print(labels, cluster_sizes, attractors)
    print("  Theta:", theta)
    print("    Cluster Set Size:", len(cluster_sizes))
    print("    Cluster Sizes:", end="")
    for i in range(len(cluster_sizes)):
        print("", cluster_sizes[i], end="")
    print("")
    nmi = normalized_mutual_info_score(true_labels, labels, average_method='geometric')
    print("    NMI Score:", nmi)
    if len(np.unique(labels)) == 3:  # We aim for 3 clusters, as suggested
        cluster_size_3_info.append([theta, cluster_sizes, nmi])
        if nmi > best_nmi:
            best_nmi = nmi

#Prints information on cluster sets with only 3 clusters (ideal)
best_nmi_info = []
print("\nInformation all on cluster sets with only 3 clusters:")
for i in range(len(cluster_size_3_info)):
    print("  Theta:", cluster_size_3_info[i][0])
    print("    Cluster Sizes:", end ="")
    for j in range(len(cluster_size_3_info[i][1])):
        print("",cluster_size_3_info[i][1][j], end="")
    print("")
    print("    NMI Score:", cluster_size_3_info[i][2])
    if best_nmi == cluster_size_3_info[i][2]:
        best_nmi_info.append([cluster_size_3_info[i][0], cluster_size_3_info[i][1]])

#Prints information on cluster set with best NMI score
print("\nInformation on cluster(s) set with best NMI score:")
print("  Best NMI Score:", best_nmi)
for i in range(len(best_nmi_info)):
    print("    Theta:", best_nmi_info[i][0])
    print("      Cluster Sizes:", end ="")
    for j in range(len(best_nmi_info[i][1])):
        print("",best_nmi_info[i][1][j], end="")
    print("")

Information all on cluster sets:
  Theta: 0.01
    Cluster Set Size: 6
    Cluster Sizes: 50 33 27 1 35 4
    NMI Score: 0.6865957545601372
  Theta: 0.030204081632653063
    Cluster Set Size: 5
    Cluster Sizes: 50 33 28 35 4
    NMI Score: 0.6931813269596764
  Theta: 0.05040816326530612
    Cluster Set Size: 4
    Cluster Sizes: 50 61 35 4
    NMI Score: 0.7386405295005112
  Theta: 0.07061224489795918
    Cluster Set Size: 4
    Cluster Sizes: 50 61 35 4
    NMI Score: 0.7386405295005112
  Theta: 0.09081632653061224
    Cluster Set Size: 4
    Cluster Sizes: 50 61 35 4
    NMI Score: 0.7386405295005112
  Theta: 0.11102040816326529
    Cluster Set Size: 4
    Cluster Sizes: 50 61 35 4
    NMI Score: 0.7386405295005112
  Theta: 0.13122448979591836
    Cluster Set Size: 4
    Cluster Sizes: 50 61 35 4
    NMI Score: 0.7386405295005112
  Theta: 0.15142857142857144
    Cluster Set Size: 4
    Cluster Sizes: 50 61 35 4
    NMI Score: 0.7386405295005112
  Theta: 0.1716326530612245
    Clust